# Sesión 4 — Métodos Cuantitativos de Pronóstico y Series de Tiempo

En este notebook reproduciremos el caso de **demanda de distribución** trabajado en la presentación. El objetivo es implementar los métodos con Python, comparar sus resultados y observar cómo las métricas de error apoyan la selección de un modelo.

Trabajaremos con:
1. Método ingenuo (Naive).
2. Promedio móvil simple.
3. Suavización exponencial simple.
4. Regresión lineal de tendencia.

## 1. Librerías

Utilizaremos `numpy`, `pandas`, `matplotlib` y herramientas de `scikit-learn` para el cálculo de la regresión y las métricas de evaluación.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Parte 1 — Datos del caso

La empresa distribuye tarimas desde su CEDIS hacia tiendas regionales. Se cuenta con la demanda real de los primeros seis meses y se desea estimar la demanda del **Mes 7**.

In [ ]:
datos = pd.DataFrame({
    "Mes": [1, 2, 3, 4, 5, 6],
    "Nombre": ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio"],
    "Demanda": [120, 135, 128, 142, 150, 158]
})

datos

## 2. Visualización de la serie

Antes de aplicar cualquier método, observaremos el comportamiento de la demanda.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(datos["Mes"], datos["Demanda"], marker="o")

plt.xticks(datos["Mes"], datos["Nombre"])
plt.xlabel("Mes")
plt.ylabel("Demanda (tarimas)")
plt.title("Demanda mensual observada")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Observación

A simple vista se aprecia un crecimiento general de la demanda, aunque existen variaciones entre meses. Esto permitirá comparar cómo reaccionan los diferentes métodos ante una serie con tendencia.

# Parte 2 — Método ingenuo (Naive)

El pronóstico de cada periodo corresponde al último valor real conocido.

In [ ]:
datos["Naive"] = datos["Demanda"].shift(1)

datos[["Mes", "Nombre", "Demanda", "Naive"]]

In [ ]:
pronostico_naive_7 = datos.loc[datos.index[-1], "Demanda"]

print(f"Pronóstico Naive para el Mes 7: {pronostico_naive_7:.1f} tarimas")

# Parte 3 — Promedio móvil simple

Utilizaremos una ventana de **3 meses**, igual que en el ejemplo de la presentación.

In [ ]:
ventana = 3

datos["Promedio_Movil_3"] = (
    datos["Demanda"]
    .rolling(window=ventana)
    .mean()
    .shift(1)
)

datos[["Mes", "Nombre", "Demanda", "Promedio_Movil_3"]]

In [ ]:
pronostico_pm_7 = datos["Demanda"].tail(ventana).mean()

print(f"Pronóstico por Promedio Móvil para el Mes 7: {pronostico_pm_7:.2f} tarimas")

# Parte 4 — Suavización exponencial simple

Aplicaremos una constante de suavización **α = 0.3**. Inicializaremos el primer pronóstico con el primer valor observado.

In [ ]:
alpha = 0.3

pronosticos_exp = [datos.loc[0, "Demanda"]]

for t in range(1, len(datos)):
    nuevo_pronostico = (
        alpha * datos.loc[t - 1, "Demanda"]
        + (1 - alpha) * pronosticos_exp[-1]
    )
    pronosticos_exp.append(nuevo_pronostico)

datos["Suavizacion_Exp"] = pronosticos_exp

datos[["Mes", "Nombre", "Demanda", "Suavizacion_Exp"]]

In [ ]:
pronostico_exp_7 = (
    alpha * datos.loc[datos.index[-1], "Demanda"]
    + (1 - alpha) * datos.loc[datos.index[-1], "Suavizacion_Exp"]
)

print(f"Pronóstico por Suavización Exponencial para el Mes 7: {pronostico_exp_7:.2f} tarimas")

# Parte 5 — Regresión lineal de tendencia

Ajustaremos una línea a los seis valores históricos utilizando el número de mes como variable explicativa.

In [ ]:
datos = pd.DataFrame({
    "Mes": [1, 2, 3, 4, 5, 6],
    "Nombre": ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio"],
    "Demanda": [5, 8, 6, 10, 15, 12]
})

X = datos[["Mes"]]
y = datos["Demanda"]

modelo_regresion = LinearRegression()
modelo_regresion.fit(X, y)

datos["Regresion"] = modelo_regresion.predict(X)

pendiente = modelo_regresion.coef_[0]
intercepto = modelo_regresion.intercept_

print(f"Ecuación estimada: Demanda = {intercepto:.2f} + {pendiente:.2f} × Mes")

In [ ]:
pronostico_regresion_7 = modelo_regresion.predict(
    pd.DataFrame({"Mes": [7]})
)[0]

print(f"Pronóstico por Regresión Lineal para el Mes 7: {pronostico_regresion_7:.2f} tarimas")

## 3. Visualización de la tendencia

La línea estimada permite observar el crecimiento que la regresión identifica en los datos.

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    datos["Mes"],
    datos["Demanda"],
    marker="o",
    label="Demanda real"
)

plt.plot(
    datos["Mes"],
    datos["Regresion"],
    marker="o",
    label="Tendencia estimada"
)

plt.scatter(
    7,
    pronostico_regresion_7,
    s=120,
    label="Pronóstico Mes 7"
)

plt.xticks(range(1, 8))
plt.xlabel("Mes")
plt.ylabel("Demanda (tarimas)")
plt.title("Regresión lineal de tendencia")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Parte 6 — Métricas de error

Para comparar correctamente los modelos necesitamos evaluar periodos donde exista tanto un valor real como un pronóstico. Calcularemos **MAE, RMSE y MAPE** utilizando únicamente las observaciones disponibles para cada método.

In [ ]:
def calcular_metricas(real, pronostico):
    real = np.asarray(real, dtype=float)
    pronostico = np.asarray(pronostico, dtype=float)

    mae = mean_absolute_error(real, pronostico)
    rmse = np.sqrt(mean_squared_error(real, pronostico))
    mape = np.mean(np.abs((real - pronostico) / real)) * 100

    return mae, rmse, mape


def evaluar_columna(columna):
    temporal = datos[["Demanda", columna]].dropna()

    return calcular_metricas(
        temporal["Demanda"],
        temporal[columna]
    )

In [ ]:
mae_naive, rmse_naive, mape_naive = evaluar_columna("Naive")
mae_pm, rmse_pm, mape_pm = evaluar_columna("Promedio_Movil_3")
mae_exp, rmse_exp, mape_exp = evaluar_columna("Suavizacion_Exp")
mae_reg, rmse_reg, mape_reg = evaluar_columna("Regresion")

resultados = pd.DataFrame({
    "Modelo": [
        "Naive",
        "Promedio Móvil (N=3)",
        "Suavización Exponencial (α=0.3)",
        "Regresión Lineal"
    ],
    "Pronóstico Mes 7": [
        pronostico_naive_7,
        pronostico_pm_7,
        pronostico_exp_7,
        pronostico_regresion_7
    ],
    "MAE": [
        mae_naive,
        mae_pm,
        mae_exp,
        mae_reg
    ],
    "RMSE": [
        rmse_naive,
        rmse_pm,
        rmse_exp,
        rmse_reg
    ],
    "MAPE (%)": [
        mape_naive,
        mape_pm,
        mape_exp,
        mape_reg
    ]
})

resultados.round(2)

### Importante

Los modelos no comienzan a generar pronósticos en el mismo periodo. Por ejemplo, un promedio móvil de tres meses necesita tres observaciones previas. Por ello, la tabla anterior es útil para explorar cada método, pero una comparación rigurosa debe realizarse sobre **los mismos periodos de evaluación**.

# Parte 7 — Comparación sobre un periodo común

Para hacer una comparación más justa, evaluaremos los cuatro métodos únicamente en los meses donde todos disponen de un pronóstico: **Meses 4, 5 y 6**.

In [ ]:
periodos_comunes = datos.loc[
    datos["Promedio_Movil_3"].notna(),
    ["Mes", "Demanda", "Naive", "Promedio_Movil_3", "Suavizacion_Exp", "Regresion"]
]

periodos_comunes

In [ ]:
comparacion_comun = []

modelos = {
    "Naive": "Naive",
    "Promedio Móvil (N=3)": "Promedio_Movil_3",
    "Suavización Exponencial (α=0.3)": "Suavizacion_Exp",
    "Regresión Lineal": "Regresion"
}

for nombre, columna in modelos.items():
    mae, rmse, mape = calcular_metricas(
        periodos_comunes["Demanda"],
        periodos_comunes[columna]
    )

    comparacion_comun.append({
        "Modelo": nombre,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape
    })

comparacion_comun = pd.DataFrame(comparacion_comun)
comparacion_comun.round(2)

# Parte 8 — Comparación visual

Representaremos los valores reales y los pronósticos generados por cada método.

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    datos["Mes"],
    datos["Demanda"],
    marker="o",
    linewidth=3,
    label="Demanda real"
)

plt.plot(datos["Mes"], datos["Naive"], marker="o", label="Naive")
plt.plot(datos["Mes"], datos["Promedio_Movil_3"], marker="o", label="Promedio móvil")
plt.plot(datos["Mes"], datos["Suavizacion_Exp"], marker="o", label="Suavización exponencial")
plt.plot(datos["Mes"], datos["Regresion"], marker="o", label="Regresión lineal")

plt.xlabel("Mes")
plt.ylabel("Demanda (tarimas)")
plt.title("Comparación de modelos de pronóstico")
plt.xticks(datos["Mes"], datos["Nombre"])
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Parte 9 — Validación temporal: entrenamiento y prueba

Hasta ahora la regresión se ajustó con todos los datos disponibles. Para mostrar una evaluación más realista, entrenaremos el modelo únicamente con los primeros meses y reservaremos los últimos periodos para comprobar qué tan bien generaliza.

In [ ]:
train = datos.iloc[:4].copy()
test = datos.iloc[4:].copy()

modelo_test = LinearRegression()
modelo_test.fit(train[["Mes"]], train["Demanda"])

test["Prediccion"] = modelo_test.predict(test[["Mes"]])

test[["Mes", "Nombre", "Demanda", "Prediccion"]]

In [ ]:
mae_test, rmse_test, mape_test = calcular_metricas(
    test["Demanda"],
    test["Prediccion"]
)

print(f"MAE: {mae_test:.2f} tarimas")
print(f"RMSE: {rmse_test:.2f} tarimas")
print(f"MAPE: {mape_test:.2f}%")

### Observación

Esta separación evita evaluar el modelo exclusivamente con los mismos datos utilizados para construirlo. En aplicaciones reales se recomienda conservar el **orden temporal** y evitar mezclar aleatoriamente observaciones pasadas y futuras.

# Parte 10 — Pronóstico y decisión logística

Finalmente, traduciremos el pronóstico del Mes 7 a una decisión operativa sencilla. Utilizaremos el pronóstico de regresión y una capacidad de **28 tarimas por camión**.

In [ ]:
import math

pronostico_operativo = math.ceil(pronostico_regresion_7)
inventario_seguridad = 5

tarimas_planificadas = pronostico_operativo + inventario_seguridad
capacidad_camion = 28

camiones = math.ceil(tarimas_planificadas / capacidad_camion)

print(f"Pronóstico Mes 7: {pronostico_operativo} tarimas")
print(f"Inventario de seguridad: {inventario_seguridad} tarimas")
print(f"Capacidad a planificar: {tarimas_planificadas} tarimas")
print(f"Camiones requeridos: {camiones}")

# Parte 11 — Ejercicio para el alumno

Una empresa registra la siguiente demanda mensual:

| Mes | Demanda |
|---:|---:|
| 1 | 210 |
| 2 | 225 |
| 3 | 218 |
| 4 | 240 |
| 5 | 255 |
| 6 | 265 |
| 7 | 258 |
| 8 | 280 |

### Actividad

1. Grafique la serie.
2. Calcule el pronóstico mediante **Naive**.
3. Calcule un **promedio móvil de 3 meses**.
4. Aplique **suavización exponencial con α = 0.4**.
5. Ajuste una **regresión lineal de tendencia**.
6. Compare los modelos mediante **MAE, RMSE y MAPE** sobre periodos comunes.
7. Pronostique el Mes 9.
8. Explique qué modelo utilizaría para apoyar la planeación de capacidad.